# Entrenamiento de modelos simples

### Carga de librerías

In [11]:
# Import necessary libraries
import os
import xarray as xr
import pandas as pd
import time
import optuna
from tabulate import tabulate
from CIMR.SurfaceWaterFraction_ATBD_main.algorithm.processing.validation_data_processing import load_lut, unravel_freqpol, atmospheric_corrections

# Modelos
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error, root_mean_squared_error

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
#from catboost import CatBoostRegressor

### Carga y preparación del dataset

In [2]:
# Open data
data_dir = os.path.join(os.getcwd(), '..', 'data')

# Load the LUT with reference land emissivities
lut_h_path = "CIMR/SurfaceWaterFraction_ATBD_main/data/lut_de_lannoy_K_h.csv"
lut_v_path = "CIMR/SurfaceWaterFraction_ATBD_main/data/lut_de_lannoy_K_v.csv"

# Try to open local dataset first, it's faster:
local_tds = "CIMR/SurfaceWaterFraction_ATBD_main/data/CIMR_SWF_TDS_JNB_v3.nc"

tds_base = xr.open_dataset(local_tds)

# NOTE: we need to normalize VOD between 0 and 1 from original LPDR's values: [0-3] Neper
min_val = 0
max_val = 3
old_attrs = tds_base.VOD.attrs
tds_base["VOD"] = (tds_base.VOD - min_val) /(max_val - min_val)

old_attrs.update(
    {
        "Valid_range": "1-0",
        "Description": "NORMALIZED VOD, original is 0-3 Neper", "Units": None
    }
)
tds_base.VOD.attrs = old_attrs

In [3]:
tds_procesado = tds_base.copy()

tds_procesado = atmospheric_corrections(tds_procesado)

tds_procesado = load_lut(tds_procesado, lut_filepath=lut_h_path)
tds_procesado = load_lut(tds_procesado, lut_filepath=lut_v_path)

# unravel tbtoa layer into frequency and polarization arrays
tds_procesado = unravel_freqpol(tds_procesado, dvars=[
    "tbtoa", "tbboa_de_lannoy"
])

# Drop the multy dimentional dvars that we dont need anymore
tds_procesado = tds_procesado.drop_vars(["tbtoa", "tbboa_de_lannoy"])

# Get emissivity from TDS, Which has ERA5 skin temperature
for freq in ["19", "37"]:
    for pol in ["H", "V"]:
        tds_procesado[f"emiss{freq}{pol}_de_lannoy"] = tds_procesado[f"tbboa_de_lannoy{freq}{pol}"] / tds_procesado["surtep_ERA5"]

# Use IGBP to remove the ocean
tds_procesado = tds_procesado.where(tds_procesado.IGBP_landcover > 0)

In [4]:
tds_ingenieria = tds_procesado.copy()

# SWF computation
ref_water_emiss_h = 0.288760

tds_ingenieria["Caracteristica1"] = (1) / (tds_ingenieria.ref_land_emis_de_lannoy_K_v - ref_water_emiss_h)
tds_ingenieria["Caracteristica2"] = (tds_ingenieria.ref_land_emis_de_lannoy_K_h) / (tds_ingenieria.ref_land_emis_de_lannoy_K_v - ref_water_emiss_h)
tds_ingenieria["Caracteristica3"] = (tds_ingenieria.emiss19H_de_lannoy) / (tds_ingenieria.ref_land_emis_de_lannoy_K_v - ref_water_emiss_h)
tds_ingenieria["SWF_calculada"] = (tds_ingenieria.ref_land_emis_de_lannoy_K_h - tds_ingenieria.emiss19H_de_lannoy) / (tds_ingenieria.ref_land_emis_de_lannoy_K_v - ref_water_emiss_h)

In [5]:
df = tds_ingenieria.to_dataframe().reset_index()

df = df.dropna(subset=['fwns'])

coords = ['lat', 'lon']   # cámbialas por tus coordenadas reales
X = df.drop(columns=['fwns'] + coords)
X = X.dropna(axis=1, how='all')
mask = X.notnull().all(axis=1)
X = X[mask]

y = df['fwns']
y = y[mask]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Entrenamiento y evaluación de modelos

In [6]:
def train_and_evaluate_models_list(X_train, y_train, X_test, y_test, models):
    results_all = {}

    for name, model in models.items() if isinstance(models, dict) else models:

        start_time = time.time()
        model.fit(X_train, y_train)
        elapsed_time = time.time() - start_time

        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)

        results = {
            "MAE_train": mean_absolute_error(y_train, y_pred_train),
            "MSE_train": mean_squared_error(y_train, y_pred_train),
            "RMSE_train": root_mean_squared_error(y_train, y_pred_train),
            "MAPE_train": mean_absolute_percentage_error(y_train, y_pred_train),
            "R²_train": r2_score(y_train, y_pred_train),

            "MAE_test": mean_absolute_error(y_test, y_pred_test),
            "MSE_test": mean_squared_error(y_test, y_pred_test),
            "RMSE_test": root_mean_squared_error(y_test, y_pred_test),
            "MAPE_test": mean_absolute_percentage_error(y_test, y_pred_test),
            "R²_test": r2_score(y_test, y_pred_test),

            "Tiempo": round(elapsed_time, 4)
        }

        results_all[name] = results
    return results_all

def format_results_table(results_dict, tablefmt="github"):
    df = pd.DataFrame(results_dict).T  # filas=modelos, columnas=métricas
    df = df.round(6)

    table_str = tabulate(df, headers="keys", tablefmt=tablefmt)

    return df, table_str

In [7]:
base_tree = DecisionTreeRegressor(max_depth=3, min_samples_leaf=10, random_state=42)

models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=0.1),
    "Lasso": Lasso(alpha=0.00001),
    "Bagging": BaggingRegressor(estimator=base_tree, n_estimators=20, random_state=42),
    "ElasticNet": ElasticNet(alpha=0.00001, l1_ratio=0.5, random_state=42),
    "MLPRegressor": MLPRegressor(hidden_layer_sizes=(128, 64, 32), max_iter=100, solver='adam', alpha=0.0001, random_state=42),
    "DecisionTree": DecisionTreeRegressor(max_depth=40, min_samples_split=20, min_samples_leaf=20 ,random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42, n_estimators=20, max_depth=40, min_samples_split=20, min_samples_leaf=20),
    "XGBoost": XGBRegressor(n_estimators=20, learning_rate=0.1, max_depth=15, verbosity=0, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=100, learning_rate=0.5, max_depth=12),
    "AdaBoost": AdaBoostRegressor(estimator=base_tree, n_estimators=40, learning_rate=0.1, random_state=42)
}

In [ ]:
results = train_and_evaluate_models_list(X_train, y_train, X_test, y_test, models)

df, table = format_results_table(results)

c:\Anaconda\envs\mi_entorno\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.034e+02, tolerance: 1.472e-01
  model = cd_fast.enet_coordinate_descent(
c:\Anaconda\envs\mi_entorno\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.045e+02, tolerance: 1.472e-01
  model = cd_fast.enet_coordinate_descent(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.024593 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5759
[LightGBM] [Info] Number of data points in the train set: 195993, number of used features: 33
[LightGBM] [Info] Start training from score 0.072311


,MAE_train,MSE_train,RMSE_train,MAPE_train,R²_train,MAE_test,MSE_test,RMSE_test,MAPE_test,R²_test,Tiempo
LinearRegression,0.020545,0.001038,0.032221,2.995916,0.861756,0.020351,0.000997,0.031573,2.019962,0.866376,0.2319
Ridge,0.020949,0.001058,0.032532,3.149838,0.859068,0.020748,0.001022,0.031967,2.107002,0.863019,0.0749
Lasso,0.021026,0.001071,0.032719,3.167497,0.857450,0.020856,0.001036,0.032191,2.124316,0.861091,3.5653
Bagging,0.025209,0.001650,0.040623,4.792106,0.780254,0.024948,0.001596,0.039956,3.078899,0.785993,16.2092
ElasticNet,0.021000,0.001069,0.032702,3.158681,0.857596,0.020824,0.001034,0.032158,2.114555,0.861376,3.7014
MLPRegressor,0.024223,0.001415,0.037615,3.934652,0.811587,0.024063,0.001388,0.037261,2.848595,0.813885,71.1051
DecisionTree,0.009844,0.000330,0.018158,1.086028,0.956094,0.011253,0.000420,0.020488,0.774533,0.943732,5.2447
RandomForest,0.009123,0.000295,0.017183,1.394699,0.960682,0.010287,0.000360,0.018974,0.932168,0.951738,74.1623
XGBoost,0.010289,0.000212,0.014545,2.356246,0.971830,0.010921,0.000246,0.015688,1.947519,0.967008,9.4682
LightGBM,0.011484,0.000292,0.017094,2.034713,0.961090,0.012252,0.000347,0.018624,1.287397,0.953503,1.4335


In [10]:
print(table)

|                  |   MAE_train |   MSE_train |   RMSE_train |   MAPE_train |   R²_train |   MAE_test |   MSE_test |   RMSE_test |   MAPE_test |   R²_test |   Tiempo |
|------------------|-------------|-------------|--------------|--------------|------------|------------|------------|-------------|-------------|-----------|----------|
| LinearRegression |    0.020545 |    0.001038 |     0.032221 |      2.99592 |   0.861756 |   0.020351 |   0.000997 |    0.031573 |    2.01996  |  0.866376 |   0.2319 |
| Ridge            |    0.020949 |    0.001058 |     0.032532 |      3.14984 |   0.859068 |   0.020748 |   0.001022 |    0.031967 |    2.107    |  0.863019 |   0.0749 |
| Lasso            |    0.021026 |    0.001071 |     0.032719 |      3.1675  |   0.85745  |   0.020856 |   0.001036 |    0.032191 |    2.12432  |  0.861091 |   3.5653 |
| Bagging          |    0.025209 |    0.00165  |     0.040623 |      4.79211 |   0.780254 |   0.024948 |   0.001596 |    0.039956 |    3.0789   |  0.785993

## Optimización de parámetros

In [12]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-2, 10, log=True),
        "tree_method": "hist"
    }

    model = XGBRegressor(**params)

    # Cross validation
    score = cross_val_score(
        model, X, y,
        scoring="neg_mean_squared_error",
        cv=3,
        n_jobs=-1
    )

    return score.mean()

# Crear el estudio
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Mejores parámetros:", study.best_params)
print("Mejor score:", study.best_value)

[I 2025-12-04 08:56:06,266] A new study created in memory with name: no-name-a76af3eb-f5f1-4599-9431-5ed5872d92e5
[I 2025-12-04 08:56:20,195] Trial 0 finished with value: -0.0014450090238824487 and parameters: {'n_estimators': 791, 'max_depth': 4, 'learning_rate': 0.22530344470211383, 'subsample': 0.775582960530508, 'colsample_bytree': 0.8220700714207487, 'gamma': 4.623758691061191, 'min_child_weight': 0.09978973860722223}. Best is trial 0 with value: -0.0014450090238824487.
[I 2025-12-04 08:56:34,795] Trial 1 finished with value: -0.0014023520052433014 and parameters: {'n_estimators': 550, 'max_depth': 6, 'learning_rate': 0.021998094627037907, 'subsample': 0.9510949609808834, 'colsample_bytree': 0.5556557265159481, 'gamma': 4.027881171327018, 'min_child_weight': 1.0866531863188378}. Best is trial 1 with value: -0.0014023520052433014.
[I 2025-12-04 08:56:42,238] Trial 2 finished with value: -0.00250338320620358 and parameters: {'n_estimators': 205, 'max_depth': 5, 'learning_rate': 0.00

Mejores parámetros: {'n_estimators': 513, 'max_depth': 6, 'learning_rate': 0.02226163005593186, 'subsample': 0.9636700044963612, 'colsample_bytree': 0.5610408461014211, 'gamma': 0.25276200039873503, 'min_child_weight': 1.392635963512697}
Mejor score: -0.0012355693809998531


In [ ]:
best_params = study.best_params
model = XGBRegressor(**best_params)

start_time = time.time()
model.fit(X_train, y_train)
elapsed_time = time.time() - start_time

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

results = {
    "MAE_train": mean_absolute_error(y_train, y_pred_train),
    "MSE_train": mean_squared_error(y_train, y_pred_train),
    "RMSE_train": root_mean_squared_error(y_train, y_pred_train),
    "MAPE_train": mean_absolute_percentage_error(y_train, y_pred_train),
    "R²_train": r2_score(y_train, y_pred_train),

    "MAE_test": mean_absolute_error(y_test, y_pred_test),
    "MSE_test": mean_squared_error(y_test, y_pred_test),
    "RMSE_test": root_mean_squared_error(y_test, y_pred_test),
    "MAPE_test": mean_absolute_percentage_error(y_test, y_pred_test),
    "R²_test": r2_score(y_test, y_pred_test),

    "Tiempo": round(elapsed_time, 4)
}

print(results)

{'MAE_train': 0.010289261117577553, 'MSE_train': 0.0002115481620421633, 'RMSE_train': 0.014544695615768433, 'MAPE_train': 2.356245517730713, 'R²_train': 0.971829891204834, 'MAE_test': 0.010921106673777103, 'MSE_test': 0.0002461238473188132, 'RMSE_test': 0.0156883355230093, 'MAPE_test': 1.9475191831588745, 'R²_test': 0.9670075178146362, 'Tiempo': 8.3146}


In [18]:
results

{'MAE_train': 0.010289261117577553,
 'MSE_train': 0.0002115481620421633,
 'RMSE_train': 0.014544695615768433,
 'MAPE_train': 2.356245517730713,
 'R²_train': 0.971829891204834,
 'MAE_test': 0.010921106673777103,
 'MSE_test': 0.0002461238473188132,
 'RMSE_test': 0.0156883355230093,
 'MAPE_test': 1.9475191831588745,
 'R²_test': 0.9670075178146362,
 'Tiempo': 8.3146}

# No ordenado

## Dataset original

In [4]:
df_base = tds_base.to_dataframe().reset_index()

df_base = df_base.dropna(subset=['fwns'])

coords = ['lat', 'lon']   # cámbialas por tus coordenadas reales
X_base = df_base.drop(columns=['fwns'] + coords)
X_base = X_base.dropna(axis=1, how='all')
mask = X_base.notnull().all(axis=1)
X_base = X_base[mask]

y_base = df_base['fwns']
y_base = y_base[mask]

X_base_train, X_base_test, y_base_train, y_base_test = train_test_split(
    X_base, y_base, test_size=0.2, random_state=42
)

In [5]:
model_base = LinearRegression()
model_base.fit(X_base_train, y_base_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [6]:
y_base_pred = model_base.predict(X_base_test)
print("R2:", r2_score(y_base_test, y_base_pred))

R2: 0.7920319245710115


## Dataset procesado

In [7]:
tds_procesado = tds_base.copy()

tds_procesado = atmospheric_corrections(tds_procesado)

tds_procesado = load_lut(tds_procesado, lut_filepath=lut_h_path)
tds_procesado = load_lut(tds_procesado, lut_filepath=lut_v_path)

# unravel tbtoa layer into frequency and polarization arrays
tds_procesado = unravel_freqpol(tds_procesado, dvars=[
    "tbtoa", "tbboa_de_lannoy"
])

# Drop the multy dimentional dvars that we dont need anymore
tds_procesado = tds_procesado.drop_vars(["tbtoa", "tbboa_de_lannoy"])

# Get emissivity from TDS, Which has ERA5 skin temperature
for freq in ["19", "37"]:
    for pol in ["H", "V"]:
        tds_procesado[f"emiss{freq}{pol}_de_lannoy"] = tds_procesado[f"tbboa_de_lannoy{freq}{pol}"] / tds_procesado["surtep_ERA5"]

# Use IGBP to remove the ocean
tds_procesado = tds_procesado.where(tds_procesado.IGBP_landcover > 0)

In [8]:
df_procesado = tds_procesado.to_dataframe().reset_index()

df_procesado = df_procesado.dropna(subset=['fwns'])

coords = ['lat', 'lon']   # cámbialas por tus coordenadas reales
X_procesado = df_procesado.drop(columns=['fwns'] + coords)
X_procesado = X_procesado.dropna(axis=1, how='all')
mask = X_procesado.notnull().all(axis=1)
X_procesado = X_procesado[mask]

y_procesado = df_procesado['fwns']
y_procesado = y_procesado[mask]

X_procesado_train, X_procesado_test, y_procesado_train, y_procesado_test = train_test_split(
    X_procesado, y_procesado, test_size=0.2, random_state=42
)

In [9]:
model_procesado = LinearRegression()
model_procesado.fit(X_procesado_train, y_procesado_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [10]:
y_procesado_pred = model_procesado.predict(X_procesado_test)
print("R2:", r2_score(y_procesado_test, y_procesado_pred))

R2: 0.8635192600991223


## Dataset con ingeniería de características

In [10]:
tds_ingenieria = tds_procesado.copy()

# SWF computation
ref_water_emiss_h = 0.288760

tds_ingenieria["Caracteristica1"] = (1) / (tds_ingenieria.ref_land_emis_de_lannoy_K_v - ref_water_emiss_h)
tds_ingenieria["Caracteristica2"] = (tds_ingenieria.ref_land_emis_de_lannoy_K_h) / (tds_ingenieria.ref_land_emis_de_lannoy_K_v - ref_water_emiss_h)
tds_ingenieria["Caracteristica3"] = (tds_ingenieria.emiss19H_de_lannoy) / (tds_ingenieria.ref_land_emis_de_lannoy_K_v - ref_water_emiss_h)
#tds_ingenieria["SWF_calculada"] = (tds_ingenieria.ref_land_emis_de_lannoy_K_h - tds_ingenieria.emiss19H_de_lannoy) / (tds_ingenieria.ref_land_emis_de_lannoy_K_v - ref_water_emiss_h)

In [11]:
df_ingenieria = tds_ingenieria.to_dataframe().reset_index()

df_ingenieria = df_ingenieria.dropna(subset=['fwns'])

coords = ['lat', 'lon']   # cámbialas por tus coordenadas reales
X_ingenieria = df_ingenieria.drop(columns=['fwns'] + coords)
X_ingenieria = X_ingenieria.dropna(axis=1, how='all')
mask = X_ingenieria.notnull().all(axis=1)
X_ingenieria = X_ingenieria[mask]

y_ingenieria = df_ingenieria['fwns']
y_ingenieria = y_ingenieria[mask]

X_ingenieria_train, X_ingenieria_test, y_ingenieria_train, y_ingenieria_test = train_test_split(
    X_ingenieria, y_ingenieria, test_size=0.2, random_state=42
)

In [12]:
model_ingenieria = LinearRegression()
model_ingenieria.fit(X_ingenieria_train, y_ingenieria_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [13]:
y_ingenieria_pred = model_ingenieria.predict(X_ingenieria_test)
print("R2:", r2_score(y_ingenieria_test, y_ingenieria_pred))

R2: 0.8663764578328497


Nota: Incluir las partes de la fórmula, incluyendo la parte lineal, aumentan la precisión más incluso que incluïr tan solo la estimación física.

## Fórmula física

In [14]:
tds_fisico = tds_procesado.copy()

ref_water_emiss_h = 0.288760
tds_fisico["SWF_calculada"] = (tds_fisico.ref_land_emis_de_lannoy_K_h - tds_fisico.emiss19H_de_lannoy) / (tds_fisico.ref_land_emis_de_lannoy_K_v - ref_water_emiss_h)

vars_to_keep = ["SWF_calculada", "fwns"]
vars_to_drop = [v for v in tds_fisico.data_vars if v not in vars_to_keep]

tds_fisico = tds_fisico.drop_vars(vars_to_drop)

In [15]:
df_fisico = tds_fisico.to_dataframe().reset_index()

df_fisico = df_fisico.dropna(subset=['fwns'])

coords = ['lat', 'lon']   # cámbialas por tus coordenadas reales
X_fisico = df_fisico.drop(columns=['fwns'] + coords)
X_fisico = X_fisico.dropna(axis=1, how='all')
mask = X_fisico.notnull().all(axis=1)
X_fisico = X_fisico[mask]

y_fisico = df_fisico['fwns']
y_fisico = y_fisico[mask]

X_fisico_train, X_fisico_test, y_fisico_train, y_fisico_test = train_test_split(
    X_fisico, y_fisico, test_size=0.2, random_state=42
)

In [16]:
model_fisico = LinearRegression()
model_fisico.fit(X_fisico_train, y_fisico_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [17]:
y_fisico_pred = model_fisico.predict(X_fisico_test)
print("R2:", r2_score(y_fisico_test, y_fisico_pred))

R2: 0.7377822552644449


### Resultados:
* Fórmula física: 0.7378
* Con los mínimos datos (tran, tbdw, tbup, VOD, vsm...): 0.792
* Con datos de corrección y lookup tables (tbboa y ref_land_emissivity...): 0.8635
* Con características basadas en la fórmula física del SWF: 0.8638

## ML complejo

| Modelo           | MAE      | RMSE     | R²       | Tiempo     |
| ---------------- | -------- | -------- | -------- | ---------- |
| RandomForest     | 0.011565 | 0.019389 | 0.842440 | 17468.8555 |
| MLPRegressor     | 0.012549 | 0.019952 | 0.833160 | 7422.0492  |
| XGBoost          | 0.013960 | 0.021999 | 0.797168 | 335.2308   |
| DecisionTree     | 0.013052 | 0.022002 | 0.797114 | 1347.4194  |
| LightGBM         | 0.014111 | 0.022554 | 0.786800 | 109.6696   |
| LinearRegression | 0.019388 | 0.031628 | 0.580758 | 56.1626    |
| AdaBoost         | 0.026933 | 0.038342 | 0.383880 | 13433.6263 |
| Bagging          | 0.025027 | 0.039292 | 0.352974 | 2539.8379  |

### RandomForest

In [13]:
model_procesado_forest = RandomForestRegressor(random_state=42, n_estimators=20, max_depth=40, min_samples_split=20, min_samples_leaf=20)
model_procesado_forest.fit(X_procesado_train, y_procesado_train)

,n_estimators,20
,criterion,'squared_error'
,max_depth,40
,min_samples_split,20
,min_samples_leaf,20
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [15]:
y_procesado_pred_forest = model_procesado_forest.predict(X_procesado_test)
print("R2:", r2_score(y_procesado_test, y_procesado_pred_forest))

R2: 0.9516581927908505


### XGBoost

In [16]:
model_procesado_boost = XGBRegressor(n_estimators=20, learning_rate=0.1, max_depth=15, verbosity=0, random_state=42)
model_procesado_boost.fit(X_procesado_train, y_procesado_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [17]:
y_procesado_pred_boost = model_procesado_boost.predict(X_procesado_test)
print("R2:", r2_score(y_procesado_test, y_procesado_pred_boost))

R2: 0.9668653607368469


In [ ]:
models = {
    "LinearRegression": LinearRegression(),
    #"Ridge": Ridge(alpha=0.1),
    #"Lasso": Lasso(alpha=0.00001),
    "Bagging": BaggingRegressor(estimator=base_tree, n_estimators=20, random_state=42),
    #"ElasticNet": ElasticNet(alpha=0.00001, l1_ratio=0.5, random_state=42),
    "MLPRegressor": MLPRegressor(hidden_layer_sizes=(128, 64, 32), max_iter=100, solver='adam', alpha=0.0001, random_state=42),
    "DecisionTree": DecisionTreeRegressor(max_depth=40, min_samples_split=20, min_samples_leaf=20 ,random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42, n_estimators=20, max_depth=40, min_samples_split=20, min_samples_leaf=20),
    #"MLPRegressor": MLPRegressor(hidden_layer_sizes=(128, 64, 32), max_iter=100, solver='adam', alpha=0.0001, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=20, learning_rate=0.1, max_depth=15, verbosity=0, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=100, learning_rate=0.5, max_depth=12),
    "AdaBoost": AdaBoostRegressor(estimator=base_tree, n_estimators=40, learning_rate=0.1, random_state=42)
}

In [12]:
def train_and_evaluate_models(X_train, y_train, X_test, y_test, model):

    start_time = time.time()
    model.fit(X_train, y_train)
    elapsed_time = time.time() - start_time

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    mae_train = mean_absolute_error(y_train, y_pred_train)
    mse_train = mean_squared_error(y_train, y_pred_train)
    rmse_train = root_mean_squared_error(y_train, y_pred_train)
    mape_train = mean_absolute_percentage_error(y_train, y_pred_train)
    r2_train = r2_score(y_train, y_pred_train)

    mae_test = mean_absolute_error(y_test, y_pred_test)
    mse_test = mean_squared_error(y_test, y_pred_test)
    rmse_test = root_mean_squared_error(y_test, y_pred_test)
    mape_test = mean_absolute_percentage_error(y_test, y_pred_test)
    r2_test = r2_score(y_test, y_pred_test)

    results = {
        "MAE_train": mae_train,
        "MSE_train": mse_train,
        "RMSE_train": rmse_train,
        "MAPE_train": mape_train,
        "R²_train": r2_train,
        "MAE_test": mae_test,
        "MSE_test": mse_test,
        "RMSE_test": rmse_test,
        "MAPE_test": mape_test,
        "R²_test": r2_test,
        "Tiempo": round(elapsed_time, 4)
    }
    return results



def format_results_table(results, tablefmt="github"):
    df = pd.DataFrame(results, index=[0]).T  # Cada métrica como fila
    df.columns = ["Valor"]
    df = df.round(6)

    table_str = tabulate(df, headers='keys', tablefmt=tablefmt)

    return df, table_str

In [13]:
model_procesado_boost = XGBRegressor(n_estimators=20, learning_rate=0.1, max_depth=15, verbosity=0, random_state=42)
results = train_and_evaluate_models(X_procesado_train, y_procesado_train, X_procesado_test, y_procesado_test, model_procesado_boost)

In [14]:
df, table = format_results_table(results)
print(table)

|            |    Valor |
|------------|----------|
| MAE_train  | 0.0103   |
| MSE_train  | 0.000213 |
| RMSE_train | 0.014584 |
| MAPE_train | 2.30259  |
| R²_train   | 0.971677 |
| MAE_test   | 0.01091  |
| MSE_test   | 0.000247 |
| RMSE_test  | 0.015722 |
| MAPE_test  | 1.92883  |
| R²_test    | 0.966865 |
| Tiempo     | 5.6912   |
